# County population by census year (IPUMS NHGIS API)

This notebook shows how to request **nominal** county-level **total population** from IPUMS NHGIS using the official Python client (`ipumspy`), load the result into pandas, save a wide CSV (one row per county, census years as columns), and plot a simple time series.

## Prerequisites

1. **NHGIS registration** (required to *submit* extracts, not just browse metadata). Log in at [NHGIS](https://www.nhgis.org/) and complete registration if prompted. If the API returns *“not registered to IPUMS nhgis”*, use [NHGIS registration / renewal](https://uma.pop.umn.edu/nhgis/registration/new).
2. An API key from [account.ipums.org/api_keys](https://account.ipums.org/api_keys), pasted into the first code cell (`IPUMS_API_KEY`).

## Data choice

NHGIS **time series table [A00](https://www.nhgis.org/time-series-tables)** (*Total Population*, **nominal** geographic integration) links comparable total-population counts across decennial censuses at the **state–county** level. Each row is an NHGIS county unit; a given census column is non-empty when that unit appears in that census (per NHGIS nominal linkage). To approximate **current** counties, we keep counties with non-missing **2020** population (you can change this rule if you prefer another baseline).

References: [IPUMS API overview](https://developer.ipums.org/docs/v2/apiprogram/), [NHGIS](https://www.nhgis.org/), [ipumspy aggregate extracts](https://ipumspy.readthedocs.io/en/latest/ipums_api/ipums_api_aggregate/index.html).

In [10]:
import os
import re
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from ipumspy import (
    AggregateDataExtract,
    IpumsApiClient,
    TimeSeriesTable,
)
from ipumspy.api.metadata import TimeSeriesTableMetadata

# Output folder (relative to Jupyter’s current working directory)
OUT_DIR = Path.cwd().resolve()
DATA_DIR = OUT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

# --- Extract scope (change in one place) ---
# List = verification run (small, minutes). None = all decennial years 1790–2020 (slow, 30–90+ min).
A00_YEARS = [2000, 2010, 2020]
# A00_YEARS = None  # uncomment for full county history

_stem = "a00_verify" if A00_YEARS else "a00_full"
ZIP_PATH = DATA_DIR / f"nhgis_county_{_stem}.zip"
CSV_OUT = DATA_DIR / f"county_population_{_stem}.csv"

# Private repo: paste your key from https://account.ipums.org/api_keys
IPUMS_API_KEY = "59cba10d8a5da536fc06b59df1d9eca2d6114582ac1c002722d948da"

client = IpumsApiClient(IPUMS_API_KEY)


## Optional: inspect metadata for table A00

Confirms available geographic levels and census years before submitting an extract.

In [11]:
# TimeSeriesTableMetadata requires both collection and table name (see ipumspy docs).
meta = client.get_metadata(TimeSeriesTableMetadata(collection="nhgis", name="A00"))
print("Description:", meta.description)
print("Geographic integration:", meta.geographic_integration)


def _pluck_names(items):
    if not items:
        return []
    if isinstance(items[0], dict):
        return [x.get("name", x.get("description", x)) for x in items]
    return list(items)


print("Geographic levels:", _pluck_names(meta.geog_levels))
years = _pluck_names(meta.years)
print("Years (count):", len(years), "| sample:", years[:5], "...", years[-3:])

Description: Total Population
Geographic integration: Nominal
Geographic levels: ['nation', 'state', 'county']
Years (count): 24 | sample: ['1790', '1800', '1810', '1820', '1830'] ... ['2000', '2010', '2020']


## Submit extract, wait, and download

If you see **`IpumsAPIAuthenticationError` ... not registered to IPUMS nhgis**, your account must be registered for NHGIS (link in the introduction). Metadata can succeed while extract submission is blocked until registration is active.

### Scope: verify first, full history later

The first code cell sets **`A00_YEARS`**. Default is **`[2000, 2010, 2020]`** — enough to test download, CSV, and plots quickly. When that works, set **`A00_YEARS = None`** for the full nominal county series (1790–2020); that job is **much** larger.

### How long does the extract take?

**Verify (3 years):** often **~5–20 minutes** (varies with queue). **Full history:** often **30–90+ minutes**. Status is printed every minute.

- **A00** at **county**: total population, nominal integration; years come from `A00_YEARS`.
- `tst_layout="time_by_column_layout"`: time points as separate columns (default).
- `data_format="csv_header"`: include the descriptive second header row (we skip it when reading into pandas).

In [ ]:
import time

if A00_YEARS is None:
    _tst = TimeSeriesTable("A00", geog_levels=["county"])
    _desc = "County A00 nominal, all census years (full)"
else:
    _tst = TimeSeriesTable("A00", geog_levels=["county"], years=A00_YEARS)
    _desc = f"County A00 nominal, years {A00_YEARS} (verify)"

extract = AggregateDataExtract(
    collection="nhgis",
    description=_desc,
    time_series_tables=[_tst],
    data_format="csv_header",
    tst_layout="time_by_column_layout",
)

submitted = client.submit_extract(extract)
print(
    f"Submitted extract #{submitted.extract_id} — {_desc}. Polling status..."
)

# wait_for_extract() is silent; poll so you see queued/started/completed
poll_sec = 60
timeout_sec = 45 * 60 if A00_YEARS else 4 * 3600  # verify ~45 min max; full up to 4 h
t0 = time.monotonic()
while True:
    status = client.extract_status(submitted)
    elapsed = time.monotonic() - t0
    print(f"[{elapsed / 60:6.1f} min] status={status!r}")
    if status == "completed":
        break
    if status == "failed":
        raise RuntimeError("Extract failed on the IPUMS server; check your account email or try a smaller request.")
    if elapsed > timeout_sec:
        raise TimeoutError(
            f"Still not completed after {timeout_sec / 3600:.1f} h — try fewer years or check https://www.nhgis.org/"
        )
    time.sleep(poll_sec)

client.download_extract(submitted, str(ZIP_PATH))
print("Saved:", ZIP_PATH)

Submitted extract #3 — waiting for IPUMS (full US county A00 is often 30–90+ min in queue)...
[   0.0 min] status='published'
[   1.0 min] status='published'


## Load CSV from the zip

NHGIS zips contain a `*_csv/` folder with one or more `.csv` files. We take the first `*_county*.csv` (or any `.csv` in that folder if needed).

In [ ]:
def load_nhgis_county_csv(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        csv_candidates = [
            n for n in names
            if n.lower().endswith(".csv") and "_csv/" in n.replace("\\", "/")
        ]
        county_csv = [n for n in csv_candidates if "county" in n.lower()]
        chosen = (county_csv or csv_candidates)[0]
        with zf.open(chosen) as f:
            # csv_header: row 0 = names, row 1 = descriptions (skip row 1)
            df = pd.read_csv(f, header=0, skiprows=[1], low_memory=False)
    return df


raw = load_nhgis_county_csv(ZIP_PATH)
raw.head()

## Wide table: identifier columns + census year columns

NHGIS names data columns with a table/series prefix and a 4-digit year suffix (e.g. `A00AA2020`). We rename those columns to the census year string. Identifier columns are those that do not match that pattern.

In [ ]:
YEAR_SUFFIX = re.compile(r"^(?P<prefix>.+?)(?P<year>\d{4})$")


def split_id_and_year_columns(columns):
    id_cols = []
    year_map = {}
    for c in columns:
        s = str(c).strip()
        m = YEAR_SUFFIX.match(s)
        if m and 1790 <= int(m.group("year")) <= 2100:
            year_map[s] = m.group("year")
        else:
            id_cols.append(s)
    return id_cols, year_map


id_cols, year_map = split_id_and_year_columns(raw.columns)
wide = raw.rename(columns=year_map)
year_cols = sorted(year_map.values(), key=int)

# Keep rows that have 2020 data = proxy for "current" county set in NHGIS nominal file
if "2020" in wide.columns:
    mask = wide["2020"].notna() & (wide["2020"].astype(str).str.strip() != "")
    wide_current = wide.loc[mask].copy()
else:
    wide_current = wide.copy()

# Numeric population columns
for y in year_cols:
    wide_current[y] = pd.to_numeric(wide_current[y], errors="coerce")

out = wide_current[id_cols + year_cols]
out.to_csv(CSV_OUT, index=False)
print("Wrote", CSV_OUT, "shape", out.shape)

## Graphical summary: U.S. total population by census year

Sum population across counties for each census column (ignoring NaNs).

In [ ]:
totals = out[year_cols].sum(skipna=True)
years_int = [int(y) for y in totals.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(years_int, totals.values, marker="o", markersize=3)
ax.set_title("U.S. total population (sum of NHGIS A00 county cells)")
ax.set_xlabel("Census year")
ax.set_ylabel("Population")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig_path = DATA_DIR / "us_total_population_timeseries.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print("Saved", fig_path)